# CycleGAN Seasonal Translation Model





## Original Paper Introducing the CycleGAN Architecture
The paper that first introduced the cycleGAN (Unpaired Image-to-Image Translation Using Cycle-Consistent Adversarial Networks) is linked here:

*   https://ieeexplore.ieee.org/document/8237506


---



## Simple Explination of CycleGAN Architecture


*   Most image-to-image translation models require paired data, where each input image has a corresponding target image (e.g., the same scene in summer and winter). This pairing provides a ground-truth reference, allowing the model to directly measure how close its output is to the correct answer and learn an accurate mapping between domains.

*   A CycleGAN removes this requirement by learning from unpaired datasets using a dual-generator, dual-discriminator architecture. It consists of two generators: one that maps images from domain A → B (e.g., summer → winter) and another that maps B → A. It also includes two discriminators, one for each domain, which learn to distinguish real images from generated ones. The generators are trained to fool the discriminators, ensuring that translated images look realistic in the target domain.


*   To ensure the translations are meaningful and not arbitrary, CycleGAN introduces a cycle consistency loss: an image translated from A → B → A should return to its original form. This constraint forces the model to preserve the underlying structure and content of the image rather than just generating any plausible out
put.


*   Because of this combination of adversarial training and cycle consistency, CycleGAN can learn to translate between domains without needing paired examples, while still maintaining consistency and realism.


---

## CycleGAN Loss Function

A CycleGAN is trained using a combination of **adversarial loss** and **cycle consistency loss**, which together allow it to learn mappings between two domains without paired data.

---

### **Full Objective**

Below is the full loss function for a CycleGAN:

<div align="center" style="font-size: 150%">

## $$ L(G, F, D_X, D_Y) = L_{GAN}(G, D_Y, X, Y) + L_{GAN}(F, D_X, Y, X) + \lambda L_{cycle}(G, F) $$

</div>

It contains 4 parts:

1. **Adversarial loss for G (X → Y)**
   - Ensures that images generated from domain X look like real images from domain Y

2. **Adversarial loss for F (Y → X)**
   - Ensures that images generated from domain Y look like real images from domain X

3. **Cycle consistency loss**
   - Ensures that translating an image to the other domain and then back again returns the original image

4. **Lambda (λ) weighting term**
   - Controls the importance of the cycle consistency loss relative to the adversarial losses  
   - A higher value places more emphasis on preserving the original image structure, while a lower value prioritizes realism in the generated images

---

### **A Closer Look At Adversarial Loss**

The adversarial loss is responsible for making generated images look realistic in the target domain. It is defined as:

<div align="center" style="font-size: 150%">

## $$
L_{GAN}(G, D_Y, X, Y) =
\mathbb{E}_{y \sim p_{data}(y)} [\log D_Y(y)] +
\mathbb{E}_{x \sim p_{data}(x)} [\log (1 - D_Y(G(x)))]
$$

</div>


This loss sets up a two-player game between the generator and the discriminator:

- The **generator (G: X → Y)** takes an image from domain X and tries to produce a version that looks like it came from domain Y  
- The **discriminator (D_Y)** looks at images and tries to determine whether they are real images from domain Y or fake images produced by the generator  

During training:
- The discriminator learns to correctly classify real images as real and generated images as fake  
- The generator learns to “fool” the discriminator by producing images that are indistinguishable from real ones  

This adversarial loss is applied in both directions:
- For **X → Y**, using generator **G** and discriminator **D_Y**
- For **Y → X**, using generator **F** and discriminator **D_X**

Overall, the adversarial loss ensures that generated images match the **visual style and distribution** of the target domain.

*Note: This is the same adversarial loss used in a standard GAN*

---

### **A Closer Look At Cycle Consistancy Loss**

The cycle consistency loss ensures that mappings between the two domains are meaningful and reversible. It is defined as:

<div align="center" style="font-size: 150%">

## $$
L_{cycle}(G, F) =
\mathbb{E}_{x \sim p_{data}(x)} \left[ \| F(G(x)) - x \|_1 \right] +
\mathbb{E}_{y \sim p_{data}(y)} \left[ \| G(F(y)) - y \|_1 \right]
$$

</div>


This enforces:
- An image from domain X is translated to domain Y using G, and then translated back to X using F, resulting in an image close to the original
- An image from domain Y is translated to domain X using F, and then translated back to Y using G, resulting in an image close to the original

In other words, if an image is translated to the other domain and then translated back, it should closely match the original image.

During training:
- The first term measures how well an image from domain X can be reconstructed after being translated to Y and back  
- The second term does the same for images from domain Y  

The use of the L1 distance encourages the reconstructed image to be **close to the original at a pixel level**, helping preserve important details such as object structure, layout, and content.

Without this loss, the model could generate images that look realistic in the target domain but do not correspond meaningfully to the input. The cycle consistency constraint prevents this by tying the two mappings together.

Overall, this loss ensures that translations are not only realistic, but also **consistent with the structure of the original image**.

---

## Implementing the CycleGAN Model in Code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Defining the Model

In [ ]:
# IMPORTS
import torch
import torch.nn as nn

# Check to make sure GPU is available
torch.cuda.is_available()

In [ ]:
###### Define the Generator model ######
class Generator(nn.Module):
    def __init__(self):
        super(Generator, self).__init__()
        # Define the layers of the generator (e.g., convolutional layers, normalization, activations)
        self.model = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
                          nn.ReLU(inplace=True),
                          nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
                          nn.ReLU(inplace=True),
                          nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1),
                          nn.Tanh()
                          )

    def forward(self, x):
        # Define the forward pass of the generator
        return self.model(x)

###### Define the Discriminator model ######
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        # Define the layers of the discriminator
        #    aka a CNN classifier that predicts if the image is real or fake
        self.model = nn.Sequential(
                          nn.Conv2d(3, 32, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1), # Reduced filters
                          nn.ReLU(inplace=True),
                          nn.Conv2d(128, 1, kernel_size=4, stride=1, padding=1),
                          nn.Sigmoid()
                          )

    def forward(self, x):
        # Define the forward pass of the discriminator
        return self.model(x)

###### Define Adversarial Loss ######
class GANLoss(nn.Module):
    def __init__(self, use_lsgan=True):
        super(GANLoss, self).__init__()
        # Use Least Squares GAN (more stable) if noted
        self.loss = nn.MSELoss() if use_lsgan else nn.BCEWithLogitsLoss()

    def forward(self, prediction, target_is_real):
        # Create target labels (1 for real, 0 for fake)
        target_tensor = torch.ones_like(prediction) if target_is_real else torch.zeros_like(prediction)

        # Compute loss between prediction and target labels
        loss = self.loss(prediction, target_tensor)
        return loss

###### Define Cycle Consistency Loss ######
def cycle_consistency_loss(real_image, reconstructed_image, lambda_weight=10.0):
    # Use L1 loss to encourage pixel-wise similarity between original and reconstructed image
    loss = nn.L1Loss()(reconstructed_image, real_image)

    # NOTE: Change lambda_weight to control strength of cycle consistancy
    return lambda_weight * loss

###### Define the full CycleGAN model ######
class CycleGAN(nn.Module):
    def __init__(self):
        super(CycleGAN, self).__init__()
        self.gen_A2B = Generator()            # Generator from domain A to domain B
        self.gen_B2A = Generator()            # Generator from domain B to domain A
        self.disc_A = Discriminator()         # Discriminator for domain A
        self.disc_B = Discriminator()         # Discriminator for domain B
        self.gan_loss = GANLoss()             # Adversarial loss
        self.lambda_cycle = 10.0              # Cycle consistency weight

    def forward(self, real_A, real_B):
        # Generate fake images
        fake_B = self.gen_A2B(real_A)    # A -> B
        fake_A = self.gen_B2A(real_B)    # B -> A
        # Reconstruct images
        cycle_A = self.gen_B2A(fake_B)   # A -> B -> A
        cycle_B = self.gen_A2B(fake_A)   # B -> A -> B

        # Return generated and reconstructed images
        return fake_A, fake_B, cycle_A, cycle_B

    def compute_loss(self, real_A, real_B):
        # Generate fake and reconstructed images
        fake_A, fake_B, cycle_A, cycle_B = self.forward(real_A, real_B)

        # Calculate the adversarial losses
        loss_G_A2B = self.gan_loss(self.disc_B(fake_B), True)
        loss_G_B2A = self.gan_loss(self.disc_A(fake_A), True)

        # Calculate cycle consistency losses
        loss_cycle_A = cycle_consistency_loss(real_A, cycle_A, self.lambda_cycle)
        loss_cycle_B = cycle_consistency_loss(real_B, cycle_B, self.lambda_cycle)

        # Total generator loss
        loss_G = loss_G_A2B + loss_G_B2A + loss_cycle_A + loss_cycle_B

        # Calculate the discriminator losses
        loss_D_A = self.gan_loss(self.disc_A(real_A), True) + self.gan_loss(self.disc_A(fake_A.detach()), False)
        loss_D_B = self.gan_loss(self.disc_B(real_B), True) + self.gan_loss(self.disc_B(fake_B.detach()), False)

        # Total discriminator loss
        loss_D = loss_D_A + loss_D_B

        return loss_G, loss_D

### Train the CycleGAN Model

To train the cycleGAN, select a dataset from the CycleGAN Datasets page (https://efrosgans.eecs.berkeley.edu/cyclegan/datasets/ )

In [ ]:
# Imports
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import os

In [ ]:
# Image plotting function to get called in testing and visualization loop
import matplotlib.pyplot as plt

def visualize_images(real_A, real_B, fake_A, fake_B):
    # Create figure for display
    plt.figure(figsize=(12, 8))

    # Arrange images for comparison
    images = [real_A, fake_B, real_B, fake_A]
    titles = ['Real A', 'Fake B', 'Real B', 'Fake A']

    for i in range(4):
        # Convert tensor to numpy image:
        #    Move to CPU (if on GPU)
        #    Rearrange from (C, H, W) → (H, W, C)
        #    Unnormalize from [-1, 1] → [0, 1] for display
        plt.subplot(2, 2, i + 1)
        plt.imshow(images[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5)  # Unnormalize
        plt.title(titles[i])
        plt.axis('off')  # Hide axis for cleaner visualization

    plt.show()

In [ ]:
# Unpaired Dataset Class Definition
class UnpairedImageDataset(Dataset):
    def __init__(self, root_A, root_B, transform=None):
        """
        Initialize the dataset with paths to domain A and domain B images.

        Args:
            root_A (str): Path to the images in domain A.
            root_B (str): Path to the images in domain B.
            transform (callable, optional): Optional transform to be applied on an image.
        """
        # Store sorted image file names for each domain
        self.A_images = sorted(os.listdir(root_A))
        self.B_images = sorted(os.listdir(root_B))

        # Store root directories
        self.root_A = root_A
        self.root_B = root_B

        # Optional image transforms
        self.transform = transform

    def __len__(self):
        return max(len(self.A_images), len(self.B_images))

    def __getitem__(self, idx):
        # Index images
        A_img_path = os.path.join(self.root_A, self.A_images[idx % len(self.A_images)])
        B_img_path = os.path.join(self.root_B, self.B_images[idx % len(self.B_images)])

        # Load images and convert to RGB
        A_img = Image.open(A_img_path).convert("RGB")
        B_img = Image.open(B_img_path).convert("RGB")

        # Apply transformations
        if self.transform:
            A_img = self.transform(A_img)
            B_img = self.transform(B_img)

        return A_img, B_img

# Define the data transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),          # Resize images to consistent size
    transforms.RandomCrop(224),             # Random crop for data augmentation
    transforms.RandomHorizontalFlip(),      # Random flip for variability
    transforms.ToTensor(),                  # Convert PIL image → PyTorch tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Initialize the dataset and dataloader
dataset = UnpairedImageDataset(root_A='/content/drive/MyDrive/Senior 1st Semester/Machine Learning/summer2winter_yosemite/trainA/', root_B='/content/drive/MyDrive/Senior 1st Semester/Machine Learning/summer2winter_yosemite/trainB/', transform=transform)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, num_workers=4)

# Initialize the CycleGAN model
cyclegan = CycleGAN()
cyclegan.to('cuda')       # Move model to GPU if available

# Define optimizers
optimizer_G = optim.Adam(list(cyclegan.gen_A2B.parameters()) + list(cyclegan.gen_B2A.parameters()), lr=0.0001, betas=(0.5, 0.999))      # Recommended learning rate: 0.001
optimizer_D = optim.Adam(list(cyclegan.disc_A.parameters()) + list(cyclegan.disc_B.parameters()), lr=0.0001, betas=(0.5, 0.999))

# Training Loop
num_epochs = 30          # Note: Recently changed from 100 to 30 to decrease training time
for epoch in range(num_epochs):
    for i, (real_A, real_B) in enumerate(dataloader):
        # Move images to GPU
        real_A = real_A.to('cuda')
        real_B = real_B.to('cuda')

        # Reset gradients before backpropogation
        optimizer_G.zero_grad()
        optimizer_D.zero_grad()

        # Compute the losses
        loss_G, loss_D = cyclegan.compute_loss(real_A, real_B)

        # Backpropagate and optimize
        # Update Generators
        loss_G.backward()
        optimizer_G.step()

        # Update Discriminators
        loss_D.backward()
        optimizer_D.step()

        # Print training progress
        if i % 100 == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Step [{i}/{len(dataloader)}], Loss_G: {loss_G.item()}, Loss_D: {loss_D.item()}')

    # Save model checkpoints periodically
    torch.save(cyclegan.state_dict(), f'cyclegan_epoch_{epoch}.pth')

### Testing and Visualizations

In [ ]:
# Load model for testing
cyclegan = CycleGAN()
cyclegan.load_state_dict(torch.load('cyclegan_epoch_29.pth'))  # load last epoch
cyclegan.to('cuda')

In [ ]:
# Testing + Visualization
cyclegan.eval()  # Set model to evaluation mode

with torch.no_grad():       # Disable gradient tracking for inference
    for i, (real_A, real_B) in enumerate(dataloader):

        if i > 5:  # Limit to a few examples
            break

        # Move inputs to GPU
        real_A = real_A.to('cuda')
        real_B = real_B.to('cuda')

        # Option 1 (not reccomended because does unncesesary computation)
        # Uses full forward pass (includes cycle reconstruction)
        # fake_A, fake_B, _, _ = cyclegan.forward(real_A, real_B)

        # Option 2 (Recommended)
        # Directly generate translated images
        fake_B = cyclegan.gen_A2B(real_A)      # A → B
        fake_A = cyclegan.gen_B2A(real_B)      # B → A

        # Visualize results
        visualize_images(real_A[0], real_B[0], fake_A[0], fake_B[0])

### Takeaways

- CycleGAN enables image-to-image translation without requiring paired datasets  
- The model uses two generators and two discriminators to learn mappings in both directions  
- Adversarial loss ensures generated images look realistic in the target domain  
- Cycle consistency loss ensures translations preserve the original image structure and content  
- The balance between realism and consistency is controlled by the lambda weighting term  
- Tuning the lambda weight is important, as higher values emphasize structure preservation while lower values prioritize visual realism  


### Other Applications of CycleGANs

- Style transfer (like turning photos into paintings or artistic styles)
- Medical imaging translation (ex: MRI ↔ CT scans)  
- Domain adaptation for machine learning (training models across different data distributions)  
- Image enhancement tasks (ex: low-light → normal lighting, blurry → sharper images)  